# N20 S10 Slope And Fit Residual Plots

This notebook reads the completed `N20x40_nsh1_dwtrunc1_C100_S10` slope-vs-cycle output and plots the fitted log-chord slope and `1 - R^2` versus cycle on logarithmic y scales.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    BUNDLE_ROOT = Path('/content/drive/MyDrive/colab_large_entanglement_scaling_N20')
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    BUNDLE_ROOT = next(
        (path / 'colab_large_entanglement_scaling_N20' for path in candidates if (path / 'colab_large_entanglement_scaling_N20').exists()),
        cwd.parent if cwd.name == 'notebooks' else cwd,
    )

RUN_ID = 'N20x40_nsh1_dwtrunc1_C100_S10'
RUN_ROOT = BUNDLE_ROOT / 'gpu_data' / 'pure_state_entanglement_slope_vs_cycle' / 'runs' / RUN_ID
FIG_ROOT = RUN_ROOT / 'figures'
FIG_ROOT.mkdir(parents=True, exist_ok=True)

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] run root: {RUN_ROOT}')

## 2. Load Completed S10 Data

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

summary_path = RUN_ROOT / 'run_summary.json'
csv_path = RUN_ROOT / 'slope_vs_cycle.csv'

if not summary_path.exists():
    raise FileNotFoundError(summary_path)
if not csv_path.exists():
    raise FileNotFoundError(csv_path)

summary = json.loads(summary_path.read_text())
if summary.get('status') != 'complete':
    raise RuntimeError(f"Run is not marked complete: status={summary.get('status')!r}")

df = pd.read_csv(csv_path).sort_values('cycle').reset_index(drop=True)
required = {'cycle', 'slope', 'slope_err', 'r2', 'target_slope'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns in {csv_path}: {sorted(missing)}')

df['one_minus_r2'] = 1.0 - df['r2']
display(Markdown(f"### Loaded `{RUN_ID}`"))
display(df.head())
display(df.tail())

## 3. Log-Scale Plots

In [ ]:
cycles = df['cycle'].to_numpy(dtype=np.int64)
slope = df['slope'].to_numpy(dtype=np.float64)
slope_err = df['slope_err'].to_numpy(dtype=np.float64)
target_slope = float(df['target_slope'].iloc[0])
one_minus_r2 = df['one_minus_r2'].to_numpy(dtype=np.float64)

positive_slope = np.isfinite(slope) & (slope > 0)
positive_residual = np.isfinite(one_minus_r2) & (one_minus_r2 > 0)

if not np.any(positive_slope):
    raise RuntimeError('No positive finite slope values available for log-scale plotting.')
if not np.any(positive_residual):
    raise RuntimeError('No positive finite 1 - R^2 values available for log-scale plotting.')

fig, ax = plt.subplots(figsize=(7.4, 4.6), constrained_layout=True)
ax.errorbar(cycles[positive_slope], slope[positive_slope], yerr=slope_err[positive_slope], marker='o', ms=3, lw=1.2, capsize=2)
ax.axhline(target_slope, color='black', ls='--', lw=1.0, label='target 1/3')
ax.set_yscale('log')
ax.set_xlabel('cycle')
ax.set_ylabel('full-x log-chord slope')
ax.set_title(f'{RUN_ID}: slope vs cycle')
ax.grid(True, which='both', alpha=0.25)
ax.legend(frameon=True)
slope_log_path = FIG_ROOT / 'slope_vs_cycle_logy.png'
fig.savefig(slope_log_path, dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7.4, 4.6), constrained_layout=True)
ax.plot(cycles[positive_residual], one_minus_r2[positive_residual], marker='o', ms=3, lw=1.2)
ax.set_yscale('log')
ax.set_xlabel('cycle')
ax.set_ylabel(r'$1 - R^2$')
ax.set_title(f'{RUN_ID}: fit residual vs cycle')
ax.grid(True, which='both', alpha=0.25)
one_minus_r2_log_path = FIG_ROOT / 'one_minus_r2_vs_cycle_logy.png'
fig.savefig(one_minus_r2_log_path, dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.7), constrained_layout=True)
axes[0].errorbar(cycles[positive_slope], slope[positive_slope], yerr=slope_err[positive_slope], marker='o', ms=3, lw=1.2, capsize=2)
axes[0].axhline(target_slope, color='black', ls='--', lw=1.0, label='target 1/3')
axes[0].set_yscale('log')
axes[0].set_xlabel('cycle')
axes[0].set_ylabel('full-x log-chord slope')
axes[0].set_title('slope')
axes[0].grid(True, which='both', alpha=0.25)
axes[0].legend(frameon=True)

axes[1].plot(cycles[positive_residual], one_minus_r2[positive_residual], marker='o', ms=3, lw=1.2)
axes[1].set_yscale('log')
axes[1].set_xlabel('cycle')
axes[1].set_ylabel(r'$1 - R^2$')
axes[1].set_title(r'$1 - R^2$')
axes[1].grid(True, which='both', alpha=0.25)
combined_log_path = FIG_ROOT / 'slope_and_one_minus_r2_vs_cycle_logy.png'
fig.suptitle(RUN_ID)
fig.savefig(combined_log_path, dpi=200)
plt.show()

print(f'[saved] {slope_log_path}')
print(f'[saved] {one_minus_r2_log_path}')
print(f'[saved] {combined_log_path}')

In [ ]:
cycles = df['cycle'].to_numpy(dtype=np.int64)
slope = df['slope'].to_numpy(dtype=np.float64)
slope_err = df['slope_err'].to_numpy(dtype=np.float64)
target_slope = 1.0 / 3.0

delta_slope = slope - target_slope
mask = (cycles > 20) & np.isfinite(delta_slope) & (delta_slope > 0)

if not np.any(mask):
    raise RuntimeError('No positive finite fitted_slope - 1/3 values available for cycles > 10.')

fig, ax = plt.subplots(figsize=(7.4, 4.6), constrained_layout=True)
ax.errorbar(
    cycles[mask],
    delta_slope[mask],
    yerr=slope_err[mask],
    marker='o',
    ms=3,
    lw=1.2,
    capsize=2,
)
ax.set_yscale('log')
ax.set_xlabel('cycle')
ax.set_ylabel(r'fitted slope $-\,1/3$')
ax.set_title(f'{RUN_ID}: fitted slope excess vs cycle')
ax.grid(True, which='both', alpha=0.25)

slope_excess_log_path = FIG_ROOT / 'slope_minus_one_third_vs_cycle_gt10_logy.png'
fig.savefig(slope_excess_log_path, dpi=200)
plt.show()

print(f'[saved] {slope_excess_log_path}')


## 4. Saved Figures

In [ ]:
display(Markdown('### Slope vs cycle, log y'))
display(Image(filename=str(slope_log_path)))
display(Markdown(r'### $1 - R^2$ vs cycle, log y'))
display(Image(filename=str(one_minus_r2_log_path)))
display(Markdown(r'### Combined'))
display(Image(filename=str(combined_log_path)))

## 5. Disconnect Runtime

In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')